# OpenEMS Wifi Antenna Simulation

This script simulates the antenna design provided in the image using OpenEMS.

**Parameters extracted from image:**
- Substrate: FR4, 1.6mm
- Trace Thickness: 35um (Standard 1oz copper)
- Dimensions:
  - Total Height H: 3.9mm
  - Trace width w: 0.3mm
  - Vertical Spacing d: 0.7mm
  - Start Width w2: 1.3mm
  - First segment height h1: 2.85mm
  - Gap s: 0.4mm
  - Bottom gap delta: 0.3mm

In [ ]:
import os
import numpy as np

# Add openEMS DLLs to path (Windows)
if os.path.exists(os.path.join(os.getcwd(), 'openEMS', 'openEMS')):
    os.add_dll_directory(os.path.join(os.getcwd(), 'openEMS', 'openEMS'))
else:
    # Fallback to system install
    try:
        os.add_dll_directory(r"C:\Users\Public\openEMS\openEMS")
    except:
        pass

from CSXCAD import ContinuousStructure
from openEMS import openEMS
from openEMS.physical_constants import *

In [ ]:
# --- Simulation Setup ---
unit = 1e-3  # all length in mm

# Frequency range of interest
f_start = 1e9     # 1 GHz
f_stop  = 6e9     # 6 GHz
f0 = 2.45e9       # Center frequency (WiFi/BLE)
fc = (f_start + f_stop) / 2

mesh_res = 0.3 # Mesh resolution (lines per mm?) No, mm step size. 0.3mm is good for 0.3mm trace width.
# A bit finer mesh for the trace
mesh_res_antenna = 0.1 

sim = openEMS(EndCriteria=1e-5)
sim.SetGaussExcite(f0, (f_stop - f_start)/2)
sim.SetBoundaryConditions(['MUR', 'MUR', 'MUR', 'MUR', 'MUR', 'MUR']) # Absorbing boundary

csx = ContinuousStructure()
FDTD = sim.GetFDTD()
FDTD.SetTimeStep(FDTD.GetTimeStep())

In [ ]:
# --- Materials ---

# Substrate (FR4)
epsilon_r = 4.4
kappa = 0.02 # Loss tangent ~0.02
substrate = csx.AddMaterial( 'substrate', epsilon=epsilon_r, kappa=kappa)

# Conductor (Copper)
# Assume perfect electric conductor (PEC) for simplicity first, or lossy metal
metal = csx.AddMetal( 'copper' ) # PEC

# --- Geometry Parameters ---
trace_width = 0.3      # w
start_width = 1.3      # w2
h_total = 3.9          # H
h1 = 2.85              # h1
spacing_d = 0.7        # d
gap_s = 0.4            # S
delta = 0.3            # delta (clearance at bottom)
pcb_thickness = 1.6    # Substrate thickness
copper_thickness = 0.035 # 35um

# Define Origin (0,0) at bottom-left corner of the trace start
start_x = 0
start_y = 0
z_metal = pcb_thickness # Top layer

# --- Build Antenna Structure ---

# 1. Wide Vertical Segment (Left)
# Bottom-Left: (0, 0), Top-Right: (w2, h1)
# Coordinates: [x1, x2], [y1, y2], [z1, z2]
metal.AddBox(priority=10, start=[start_x, start_y, z_metal], stop=[start_x + start_width, start_y + h1, z_metal])

# 2. Top Horizontal Bar
# The image shows vertical fingers connected by a top bar.
# Or is it a meander?
# Let's assume it looks like an Inverted-F or Comb structure.
# Based on standard PCB antennas, meander monopoles weave.
# BUT, look at the top. The top bar is continuous.
# It connects the wide segment to the first 4-5 thin vertical segments.
# Let's trace carefully: 
# Wide segment goes UP.
# Top bar goes RIGHT.
# Then several vertical segments go DOWN from the top bar.
# Let's assume the top bar connects them all.

# Let's calculate the X positions of vertical segments
# Segment 1 (Wide): x=0 to 1.3
# Gap S = 0.4
# Segment 2: x = 1.3 + 0.4 = 1.7. Width w=0.3. So x=1.7 to 2.0
# Spacing d=0.7 (center-to-center? or gap?). 
# The label d=0.7 is between the *centers* of the thin lines usually, or the repeating unit.
# If unit d=0.7, then next is at x = 1.7 + 0.7? Or is 0.7 the gap?
# Let's look at the arrows. The arrows for d=0.7 span from one left edge to next left edge. So Pitch = 0.7mm.
# Wait, the arrows for d=0.7 are between the vertical lines.
# Let's assume Pitch = 0.7 mm.
# So:
#   Line 2 Start X = 1.7
#   Line 3 Start X = 1.7 + 0.7 = 2.4
#   Line 4 Start X = 2.4 + 0.7 = 3.1
#   Line 5 Start X = 3.1 + 0.7 = 3.8
#   Line 6 Start X = 3.8 + 0.7 = 4.5
#   Line 7 Start X = 4.5 + 0.7 = 5.2

# How many lines? The image shows:
# Wide block
# Then 1, 2, 3, 4, 5 thin lines.
# The Top bar connects them all?
# The Bottom bar connects some? NO, bottom is open for most.
# It looks like a Meander Meandered Line.
# UP (Wide), RIGHT (Top), DOWN (Thin 1), RIGHT (Bottom), UP (Thin 2), RIGHT (Top), DOWN (Thin 3)...
# Let's look at the top and bottom connections.
# The wide block goes up.
# Top horizontal line connects Wide to Thin 1.
# Thin 1 goes down.
# Bottom horizontal line connects Thin 1 to Thin 2.
# Thin 2 goes up.
# Top horizontal line connects Thin 2 to Thin 3.
# ... It's a snake.

# Let's re-examine image.
# Top edge is straight. A continuous top bar? No, look at the gaps in the metal.
# Ah, the red is metal. The grey is substrate.
# Correct interpretation: Meander Snake.
# 1. UP (Wide)
# 2. RIGHT (Top) -> Connects to Thin 1
# 3. DOWN (Thin 1)
# 4. RIGHT (Bottom) -> Connects to Thin 2
# 5. UP (Thin 2)
# 6. RIGHT (Top) -> Connects to Thin 3
# 7. DOWN (Thin 3)
# ... and so on.

# Let's list the segments:
# Segment 0 (Wide): x=[0, 1.3], y=[0, 2.85] (starts at y=0?)
#   Wait, Left segment is labeled h1=2.85mm. Total H=3.9mm.
#   So top of Left Segment is at y = 2.85?
#   Or does it start higher? 
#   The top line is at y = H = 3.9? No. H=3.9 is the total height of the PCB area?
#   The dimension H=3.9mm arrow goes from bottom yellow line to top yellow line.
#   The dimension h1=2.85mm arrow goes from bottom yellow line to the top of the wide block.
#   So Wide Block top is at y = 2.85.
#   The Meander starts from the top of the wide block?
#   Actually, let's look at the top part. There is a gap between 2.85 and 3.9?
#   Or is the top wire at y=3.9? 
#   Wait, h1=2.85 is the height of the wide block. 
#   The meander turns seem to go up to H=3.9?
#   Let's check the top horizontal segments. They align with the top of the wide block?
#   NO. The top of the wide block aligns with the top of the meander loops.
#   The H=3.9 seems to be the total clearance area height. 
#   Wait, let's look at h1 again. It measures the vertical length of the wide strip.
#   And the thin strips seem to have same top Y coordinate.
#   So Top Y = 2.85 + Y_start.
#   Let's assume the bottom of the wide strip is at Y=0. Then Top Y = 2.85.
#   The dimension w=0.3mm at the top (with arrows pointing to the horizontal bar thickness).
#   So the horiz bars are 0.3mm thick.
#   The dimension H=3.9mm is mysterious. Maybe distance to ground plane edge? 
#   But the red shape is only ~2.85mm high.
#   Ah, look at delta=0.3mm at the bottom.
#   Maybe the antenna starts at Y = delta = 0.3?
#   And H = 3.9 is the total size of the cutout in the ground plane? 
#   Usually these are clearance area definitions.
#   Let's assume the Antenna Metal is bounded by:
#     Y_bottom = delta (or 0 relative to feed). 
#     Y_top = ???
#     The height of the metal part is what matters.
#     Let's assume the metal shape starts at y=0 and goes up to y=2.85.
#     (Ignoring the H=3.9 for geometric construction of the metal itself, treating it as the keepout height).

# Coordinate Construction (assuming bottom-left of wide block is (0,0)):
#   Wide Block: Box(0, 0, 1.3, 2.85)
#   Thickness of horiz lines = 0.3 (labeled 'w=0.3mm' at top and bottom)
#   
#   Connection 1 (Top): From x=1.3 to x=1.7 (Gap S=0.4). 
#      Rect: x=[1.3, 1.7], y=[2.85-0.3, 2.85]? Or y=[2.85, 2.85+0.3]?
#      Usually corners are flush. Let's assume top aligns with 2.85.
#      So Horizontal Top Bar 1: x=[1.3, 1.7+width], y=[2.55, 2.85]
#      Wait, the gap is S=0.4. The next line starts at x = 1.3+0.4 = 1.7.
#      The vertical line width is 0.3. So Line 1 is x=[1.7, 2.0].
#      So Top Bar connects x=1.3 to 1.7 (and overlaps slightly for continuity).
#      Let's define segments by centerlines or blocks.

#   Let's use a function to draw the snake.
#   Current Point (CP): Top-Right of Wide Block -> (1.3, 2.85)
#   Step 1: Go Right by S=0.4. Draw Horiz Bar. 
#      Block x=[1.3, 1.7+0.3], y=[2.85-0.3, 2.85]. (Assuming top-aligned)
#      New X = 1.7. This is the left edge of vertical line 1.
#   Step 2: Vertical Line 1. Go Down.
#      Block x=[1.7, 2.0], y=[y_bottom, 2.85].
#      What is y_bottom? 
#      The bottom horizontal segments are also width 0.3.
#      Do they go down to y=0? 
#      The wide block starts at y=0.
#      The thin meanders seem to align with the bottom of the wide block.
#      So y_bottom = 0.
#      So Vert Line 1: x=[1.7, 2.0], y=[0, 2.85-0.3] (overlap top) or y=[0.3, 2.55]?
#      Let's just overlap: y=[0, 2.85].
#   Step 3: Go Right at Bottom. 
#      Spacing to next line. Pitch d=0.7. 
#      Next line starts at 1.7 + 0.7 = 2.4.
#      Gap between line 1 (ends at 2.0) and line 2 (starts at 2.4) is 0.4.
#      Wait, S=0.4, d=0.7. Pitch 0.7 - Width 0.3 = Space 0.4. Matches.
#      So uniform spacing.
#      Bot Bar 1: x=[1.7, 2.4+0.3], y=[0, 0.3]. (Assuming bottom-aligned)
#   Step 4: Vertical Line 2. Go Up.
#      Line 2 starts at x=2.4. Width 0.3 -> x=[2.4, 2.7].
#      y=[0, 2.85].
#   Step 5: Go Right at Top.
#      Top Bar 2: x=[2.4, 3.1+0.3], y=[2.55, 2.85].
#      
#   Repeat.
#
#   Let's count the verticals:
#   Wide Block (0)
#   Thin 1 (Down)
#   Thin 2 (Up)
#   Thin 3 (Down)
#   Thin 4 (Up)
#   Thin 5 (Down)
#   Is there a Thin 6? 
#   Image shows: Wide --(top)--> V1 --(bot)--> V2 --(top)--> V3 --(bot)--> V4 --(top)--> V5 --(bot)--> V6?
#   Let's look at the right edge.
#   There is a vertical line at the end.
#   Counting 'd' intervals: There are 6 dimension lines for d=0.7.
#   So there are 6 gaps/pitches.
#   So there are 1 (start) + 6 (thin) lines? Or 1 wide + 6 thin lines?
#   Let's assume 1 Wide + 6 Thin lines.
#   Connectivity Sequence:
#   Wide (Up) -> Top -> V1 (Down) -> Bot -> V2 (Up) -> Top -> V3 (Down) -> Bot -> V4 (Up) -> Top -> V5 (Down) -> Bot -> V6 (Up)?
   
   
pass

In [ ]:
# Define Geometry Helper
def add_rect(x_start, x_end, y_start, y_end):
    metal.AddBox(priority=10, start=[x_start, y_start, z_metal], stop=[x_end, y_end, z_metal])

# Current Cursor
cur_x = start_x               # 0
cur_y = 0

# 1. Wide Block
add_rect(cur_x, cur_x + start_width, 0, h1)
cur_x += start_width          # 1.3

# 2. Meander Loop
num_thin_lines = 6 
width = trace_width           # 0.3
gap = spacing_d - width       # 0.7 - 0.3 = 0.4. Matches S=0.4. Good.

is_at_top = True # We are at top of Wide Block (Effective)

for i in range(num_thin_lines):
    # Calculate x for this line
    # Gap first
    gap_start_x = cur_x
    line_start_x = cur_x + gap
    line_end_x = line_start_x + width
    
    # Draw Connecting Horizontal Bar
    if is_at_top:
        # Draw top bar from Prev Line Right Edge to Current Line Right Edge (to cover corner)
        # y range: [h1 - width, h1]
        # x range: [cur_x - width_of_prev?, line_end_x]
        # Actually, simpler: rectangle covering the gap and overlaps.
        # Connect: (cur_x, h1) to (line_start_x, h1)
        add_rect(cur_x - 0.01, line_end_x, h1 - width, h1) # Overlap slightly left
    else:
        # Draw bottom bar
        add_rect(cur_x - 0.01, line_end_x, 0, width)
        
    # Draw Vertical Line
    # y range: [0, h1]
    add_rect(line_start_x, line_end_x, 0, h1)
    
    # Update State
    cur_x = line_end_x
    is_at_top = not is_at_top


# --- Ground Plane ---
# The image shows a grey box around the antenna. This is the clearance area.
# The GND is outside this box.
# Let's define a large GND plane with a cutout.
# Cutout Size:
#   Height H_clearance = 3.9mm
#   Width W_clearance = cur_x + some margin? 
#   Let's assume the clearance is tight to the antenna on top/right, 
#   and the antenna connects to GND on the left?
#   Actually, the Wide Block is likely the Feed Line or Monopole Element.
#   Usually Monopoles are fed against ground.
#   Let's assume a Microstrip Feed.
#   Feed Layout:
#     A 50-ohm line coming from bottom, connecting to the Wide Block?
#     Image shows pads "1 GND", "ANT_OUT", "GND".
#     This suggests a Copy-Planar Waveguide (CPW) or just a connector footprint.
#     The "ANT_OUT" pad is likely the feed point.
#     It connects to the Wide Block.
#     So the Wide block *is* the radiator start.
#     The Ground Plane is everywhere *except* the antenna area.
#     Let's define the PCB size first.

gnd_w = 40
gnd_h = 40
substrate_box = [-10, -10, 0, gnd_w, gnd_h, pcb_thickness]
csx.AddBox(priority=0, material=substrate, start=[-10, -10, 0], stop=[30, 30, pcb_thickness]) # Arbitrary PCB size

# Ground Plane (Bottom Layer? Or Top Layer with cutout?)
# Image shows traces on Top. Grey background usually means substrate (FR4). 
# If it's a MIFA (Meandered Inverted F), the ground is usually on the same layer (Top) next to it,
# OR on the bottom layer with a cutout.
# Given "GND" pads on the same layer, it's likely a Top-Layer Ground Poly with a cutout.
# Let's generate a Top Layer Ground Pour with a clear rectangle for the antenna.

# Clearance Rect:
#   X: From 0 (or slightly left) to End of Antenna.
#   Y: From 0 to H=3.9.
#   The antenna sits inside this void.
#   Wait, the wide block connects to the pad.
#   Let's assume the Ground Plane starts at Y < 0 (Connector area) and Y > 0 (Side grounds).
#   Let's modify the Ground Geometry:
#     1. Full Ground Plane on Bottom? (Common for monopoles inverted F)
#     2. OR Coplanar Ground on Top?
#     MIFA usually has a ground leg. 
#     If this is a simple Monopole, it just needs a ground reference.
#     Let's assume a Ground Plane on the Bottom Layer (z=0) for simplicity, 
#     OR a Top Ground with Cutout.
#     The labels "GND" "ANT_OUT" "GND" bottom suggest a CPW-ish launcher.
#     Let's put a Ground Plane on the Top Layer, with a big rectangular Hole.
#     Hole: x=[-1, antenna_width+1], y=[0, 3.9].
#     Wait, the antenna connects to ANT_OUT. The wide block is the feed.
#     So the feed line must be isolated.
#     Let's assume the main Ground Plane is slightly below Y=0 (the feed line area).
#     Actually, typically for these PCB antennas:
#       - Antenna is in a "Clearance Zone" (No copper on Top or Bottom).
#       - Ground Plane is on the rest of the PCB.
#       - Feed point is at the edge of the Ground Plane.
#   
#   Proposed Model:
#     - Substrate: Infinite (large)
#     - Ground Plane: Large Rectangle on Top Layer (z_metal), but subtract the Antenna Region.
#     - Antenna Region Box: x=[0, 5.5], y=[0, 3.9].
#     - Actually, let's just make the ground plane start at x < 0 and x > 5.5, and y < 0?
#     - No, usually ground is everywhere.
#     - Let's create a "Keepout" box and subtract it from a full metal plane?
#       CSXCAD doesn't do boolean subtract easily on primitives. Better to draw Ground Rects around it.
#   
#   Let's add Ground Plane Rectangles:
#     1. Left Ground: x=[-10, -0.5], y=[-10, 10]
#     2. Right Ground: x=[antenna_end_x + 0.5, 20], y=[-10, 10]
#     3. Bottom Ground: x=[-0.5, antenna_end_x + 0.5], y=[-10, 0] (Below the antenna)
#     4. Top Ground? Usually open at top for radiation. Let's leave top open.
#
#   Feed:
#     Lumped Port between the Bottom edge of the Wide Block and the Bottom Ground Plane.
#     Gap = delta? No, the wide block goes to y=0? 
#     Let's assume the Wide Block starts at y=0, and the Ground Plane ends at y=0 (or y=-gap).
#     Let's assume a gap of 0.5mm for the feed port.
#     So Ground Plane ends at y=0. Antenna starts at y=0.
#     Port is a voltage source across a small gap? 
#     Actually, if it's a Monopole, the port is between the base of the monopole and ground.
#     Let's place a Lumped Port at (x=0 to 1.3, y=0, z=z_metal). 
#     Connecting to what? Virtual ground? 
#     Better: Simulate with a small gap in Y.
#     Antenna starts at y = 0.5.
#     Ground ends at y = 0.
#     Port spans y=[0, 0.5].

# Let's refine based on the picture labels.
# "delta = 0.3mm" is at the right bottom edge.
# It indicates the clearance between the antenna meander and the ground plane below it.
# So there IS a ground plane below the meander part.
# So Ground is at y < 0.
# Antenna Meander Bottom is at y = delta = 0.3.
# Wide Block Bottom? 
# The Wide Block seems to go lower to connect to the pad.
# Let's assume Wide Block goes to y=0.
# Meander Loops go down to y=0.3.
# Ground Plane is at y <= 0.
# Feed is at the Wide Block Base (y=0) to Ground (y=0)? 
# This implies the Wide Block is connected to the Feed Pad, which is separated from GND pads.
# Let's model the Feed Gap.
# Let's assume the Feed Source is at the bottom of the Wide Block.
# We will add a resistor (50 ohm) + Excitation in a small gap.
# Let's move the Ground Plane to y = -0.5.
# Antenna starts at y = 0.
# Port at y = [-0.5, 0].

# WAIT. The "delta=0.3" is the gap between the meander and the GROUND POURED AREA.
# This confirms Ground is surrounding the antenna.

# Revised Plans:
#   Ground Plane: 
#     Rectangle y < 0 (Bottom area).
#   Antenna:
#     Wide Block starts at y=0 (connected to feed).
#     Meander Bottoms at y = delta = 0.3 (clearing the ground).
#     So the Wide block is longer than the meander fingers.
#   Feed:
#     Port from Wide Block (y=0) to Ground (y=0). 
#     Since they touch, we need a gap.
#     Let's cut a small slot in the ground for the feed, or rely on the pad spacing.
#     Simpler Model: Vertical Lumped Field Excitation.
#     Let's say Ground is at y=0. Antenna feed starts at y=0.5.
#     Connect with 50 ohm port.

# Final Geometry Plan for Script:
# 1. Substrate Box: (-10,-10) to (20, 10).
# 2. Ground Plane (Metal): 
#      Rect 1: (-10, -10) to (20, 0). (Bottom Ground)
#      (Maybe side grounds too if structure implies, but MIFA often just has ground on one side)
# 3. Antenna (Metal):
#      Shift everything up by `feed_gap = 0.5`.
#      Wide Block: x=[0, 1.3], y=[feed_gap, feed_gap + h1].
#      Meander Bottom y: feed_gap + delta (0.3).  => Bottom of thin loops is `feed_gap + 0.3`.
#      Meander Top y: feed_gap + h1 (assuming aligned with top of wide block).
# 4. Feed Port:
#      Lumped Port from [0, 0, z] to [0, feed_gap, z]. (Vertical E-field).
#      Resistance = 50 Ohm.

pass

In [ ]:
# Export to script file